# Ray cluster autoscaling lifecycle

This notebook creates a Ray cluster with in-tree autoscaling, submits a bursty CPU workload,
observes workers scale up and back down, then tears the cluster down.

> **Kueue:** Autoscaling is not supported when Kueue manages RayClusters in your namespace.
Use a non-Kueue project or see [RHAIRFE-909](https://redhat.atlassian.net/browse/RHAIRFE-909).


In [ ]:
import time

from kubernetes import client as k8s_client

from codeflare_sdk import Cluster, ClusterConfiguration, TokenAuthentication
from codeflare_sdk.common.kubernetes_cluster import get_api_client


In [ ]:
# Authenticate the CodeFlare SDK
# On OpenShift, retrieve the token with `oc whoami -t` and the server with `oc whoami --show-server`.
auth = TokenAuthentication(
    token='',
    server='',
    skip_tls=False,
)
auth.login()


Configure an autoscaling Ray cluster. The SDK sets `enableInTreeAutoscaling` on the RayCluster
and maps `min_workers` / `max_workers` to worker replica bounds.

NOTE: If running outside of RHOAI workbench pods, set `namespace="rhods-notebooks"` (or your project namespace).


In [ ]:
CLUSTER_NAME = 'ray-autoscale'
MIN_WORKERS = 1
MAX_WORKERS = 2
LOAD_TASKS = 3
LOAD_SLEEP_S = 180

cluster = Cluster(
    ClusterConfiguration(
        name=CLUSTER_NAME,
        enable_autoscaling=True,
        min_workers=MIN_WORKERS,
        max_workers=MAX_WORKERS,
        head_cpu_requests=1,
        head_cpu_limits=1,
        head_memory_requests=7,
        head_memory_limits=8,
        worker_cpu_requests=1,
        worker_cpu_limits=1,
        worker_memory_requests=5,
        worker_memory_limits=6,
        head_extended_resource_requests={'nvidia.com/gpu': 0},
        worker_extended_resource_requests={'nvidia.com/gpu': 0},
    )
)


In [ ]:
# Create the Ray cluster
cluster.apply()


In [ ]:
cluster.wait_ready()


In [ ]:
cluster.details()


Helper functions below count worker pods and poll until the expected replica count is reached.
Use the `oc` commands to inspect RayCluster status and worker pods while the test runs.


In [ ]:
def count_workers(cluster_name: str, namespace: str) -> int:
    api = k8s_client.CoreV1Api(get_api_client())
    label = f'ray.io/node-type=worker,ray.io/cluster={cluster_name}'
    pods = api.list_namespaced_pod(namespace, label_selector=label)
    return len(pods.items or [])


def wait_for_worker_count(
    cluster_name: str,
    namespace: str,
    predicate,
    timeout_s: int = 900,
    poll_s: int = 10,
) -> int:
    start = time.time()
    last = None
    while time.time() - start < timeout_s:
        last = count_workers(cluster_name, namespace)
        print(f'Worker count: {last}')
        if predicate(last):
            return last
        time.sleep(poll_s)
    raise TimeoutError(
        f'Timed out waiting for worker count. cluster={cluster_name} last={last}'
    )


NAMESPACE = cluster.config.namespace
print(f'Namespace: {NAMESPACE}')
!oc get raycluster {CLUSTER_NAME} -n {NAMESPACE} -o yaml | grep -E 'enableInTreeAutoscaling|minReplicas|maxReplicas|replicas' || true

initial_workers = wait_for_worker_count(
    CLUSTER_NAME, NAMESPACE, lambda n: n == MIN_WORKERS, timeout_s=600
)
print(f'Initial worker count matches min_workers={MIN_WORKERS}: {initial_workers}')


Submit a CPU-bound Ray job that queues more tasks than the cluster can run at the current size.
With `max_workers=2`, three tasks at 1 CPU each should trigger a scale-up to two workers.


In [ ]:
client = cluster.job_client

submission_id = client.submit_job(
    entrypoint=(
        f'AUTOSCALING_TASKS={LOAD_TASKS} AUTOSCALING_TASK_SLEEP_S={LOAD_SLEEP_S} '
        'python autoscaling_load.py'
    ),
    runtime_env={'working_dir': './'},
)
print(f'Submitted autoscaling load job: {submission_id}')
print(f'Job status: {client.get_job_status(submission_id)}')


In [ ]:
scaled_up = wait_for_worker_count(
    CLUSTER_NAME,
    NAMESPACE,
    lambda n: n >= MAX_WORKERS,
    timeout_s=900,
)
print(f'Scale-up observed: {scaled_up} worker(s) (max_workers={MAX_WORKERS})')
!oc get pods -n {NAMESPACE} -l ray.io/cluster={CLUSTER_NAME},ray.io/node-type=worker


Wait for the load job to finish, then wait for the autoscaler to scale back down to `min_workers`.
Scale-down can take several minutes after the cluster becomes idle.


In [ ]:
while client.get_job_status(submission_id) in {'PENDING', 'RUNNING'}:
    print(f'Job status: {client.get_job_status(submission_id)}')
    time.sleep(15)

print(f'Final job status: {client.get_job_status(submission_id)}')
print(client.get_job_logs(submission_id))

scaled_down = wait_for_worker_count(
    CLUSTER_NAME,
    NAMESPACE,
    lambda n: n == MIN_WORKERS,
    timeout_s=900,
)
print(f'Scale-down observed: {scaled_down} worker(s) (min_workers={MIN_WORKERS})')


In [ ]:
client.delete_job(submission_id)
cluster.down()
